In [ ]:
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [ ]:

df = pd.read_csv('02-23-2018.csv')
print(df.head())

   Dst Port  Protocol            Timestamp  Flow Duration  Tot Fwd Pkts  \
0        22         6  23/02/2018 08:18:29        1532698            11   
1       500        17  23/02/2018 08:17:45      117573855             3   
2       500        17  23/02/2018 08:17:45      117573848             3   
3        22         6  23/02/2018 08:19:55        1745392            11   
4       500        17  23/02/2018 08:18:17       89483474             6   

   Tot Bwd Pkts  TotLen Fwd Pkts  TotLen Bwd Pkts  Fwd Pkt Len Max  \
0            11             1179             1969              648   
1             0             1500                0              500   
2             0             1500                0              500   
3            11             1179             1969              648   
4             0             3000                0              500   

   Fwd Pkt Len Min  ...  Fwd Seg Size Min  Active Mean  Active Std  \
0                0  ...                32          0.0    

In [ ]:
df.value_counts('Label')

,count
Label,
Benign,1048009
Brute Force -Web,362
Brute Force -XSS,151
SQL Injection,53


In [ ]:
df.count()

,0
Dst Port,1048575
Protocol,1048575
Timestamp,1048575
Flow Duration,1048575
Tot Fwd Pkts,1048575
...,...
Idle Mean,1048575
Idle Std,1048575
Idle Max,1048575
Idle Min,1048575


##Preprocessing

dropping timestamp column

In [ ]:
df.drop(['Timestamp'], axis=1, inplace=True)

In [ ]:
df.isnull().sum()

,0
Dst Port,0
Protocol,0
Flow Duration,0
Tot Fwd Pkts,0
Tot Bwd Pkts,0
...,...
Idle Mean,0
Idle Std,0
Idle Max,0
Idle Min,0


##protocol column is hot encoded

In [ ]:
df['Protocol'].unique()

array([ 6, 17,  0])

In [ ]:
df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')
df.head()

,Dst Port,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Proto_0,Proto_6,Proto_17
0,22,1532698,11,11,1179,1969,648,0,107.181818,196.245162,...,0,0,0.0,0.000000e+00,0,0,Benign,False,True,False
1,500,117573855,3,0,1500,0,500,500,500.000000,0.000000,...,0,0,58786927.5,2.375324e+07,75583006,41990849,Benign,False,False,True
2,500,117573848,3,0,1500,0,500,500,500.000000,0.000000,...,0,0,58786924.0,2.375325e+07,75583007,41990841,Benign,False,False,True
3,22,1745392,11,11,1179,1969,648,0,107.181818,196.245162,...,0,0,0.0,0.000000e+00,0,0,Benign,False,True,False
4,500,89483474,6,0,3000,0,500,500,500.000000,0.000000,...,4000364,4000364,21370777.5,1.528092e+07,41989576,7200485,Benign,False,False,True


##Feature Scaling

In [ ]:
cols = df.columns[1:] # Skip Dst Port
cols = df.columns.drop('Label')
print(df[cols].isnull().sum())      # Check for NaNs
print((df[cols] == np.inf).sum())   # Check for +inf
print((df[cols] == -np.inf).sum())  # Check for -inf

Dst Port           0
Flow Duration      0
Tot Fwd Pkts       0
Tot Bwd Pkts       0
TotLen Fwd Pkts    0
                  ..
Idle Max           0
Idle Min           0
Proto_0            0
Proto_6            0
Proto_17           0
Length: 80, dtype: int64
Dst Port           0
Flow Duration      0
Tot Fwd Pkts       0
Tot Bwd Pkts       0
TotLen Fwd Pkts    0
                  ..
Idle Max           0
Idle Min           0
Proto_0            0
Proto_6            0
Proto_17           0
Length: 80, dtype: int64
Dst Port           0
Flow Duration      0
Tot Fwd Pkts       0
Tot Bwd Pkts       0
TotLen Fwd Pkts    0
                  ..
Idle Max           0
Idle Min           0
Proto_0            0
Proto_6            0
Proto_17           0
Length: 80, dtype: int64


In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)
df[cols] = df[cols].fillna(df[cols].mean())

In [ ]:
df_full = df
df_full.head()
cols_full = df_full.columns[1:] # Skip Dst Port
cols_full = df_full.columns.drop('Label')

minMaxScaler = MinMaxScaler().fit(df_full[cols_full])  # Learn min and max
df_full[cols_full] = minMaxScaler.transform(df_full[cols_full])   # Apply scaling (0 to 1)
df_full.head()

,Dst Port,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Proto_0,Proto_6,Proto_17
0,0.000336,0.012772,0.000172,0.000089,0.000745,0.000013,0.098301,0.000000,0.059231,0.139603,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Benign,0.0,1.0,0.0
1,0.007630,0.979782,0.000034,0.000000,0.000948,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.000000,0.000000,0.489925,0.331849,0.629902,0.349948,Benign,0.0,0.0,1.0
2,0.007630,0.979782,0.000034,0.000000,0.000948,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.000000,0.000000,0.489925,0.331849,0.629902,0.349948,Benign,0.0,0.0,1.0
3,0.000336,0.014545,0.000172,0.000089,0.000745,0.000013,0.098301,0.000000,0.059231,0.139603,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Benign,0.0,1.0,0.0
4,0.007630,0.745696,0.000086,0.000000,0.001896,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.035348,0.035348,0.178102,0.213485,0.349938,0.060008,Benign,0.0,0.0,1.0


In [ ]:
df = df[df['Label'] == 'Benign']
df.head()

,Dst Port,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Proto_0,Proto_6,Proto_17
0,0.000336,0.012772,0.000172,0.000089,0.000745,0.000013,0.098301,0.000000,0.059231,0.139603,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Benign,0.0,1.0,0.0
1,0.007630,0.979782,0.000034,0.000000,0.000948,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.000000,0.000000,0.489925,0.331849,0.629902,0.349948,Benign,0.0,0.0,1.0
2,0.007630,0.979782,0.000034,0.000000,0.000948,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.000000,0.000000,0.489925,0.331849,0.629902,0.349948,Benign,0.0,0.0,1.0
3,0.000336,0.014545,0.000172,0.000089,0.000745,0.000013,0.098301,0.000000,0.059231,0.139603,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Benign,0.0,1.0,0.0
4,0.007630,0.745696,0.000086,0.000000,0.001896,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.035348,0.035348,0.178102,0.213485,0.349938,0.060008,Benign,0.0,0.0,1.0


In [ ]:
cols = df.columns[1:] # Skip Dst Port
cols = df.columns.drop('Label')

minMaxScaler = MinMaxScaler().fit(df[cols])  # Learn min and max
df[cols] = minMaxScaler.transform(df[cols])   # Apply scaling (0 to 1)
df.head()

,Dst Port,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Proto_0,Proto_6,Proto_17
0,0.000336,0.012772,0.000172,0.000089,0.000745,0.000013,0.098301,0.000000,0.059231,0.139603,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Benign,0.0,1.0,0.0
1,0.007630,0.979782,0.000034,0.000000,0.000948,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.000000,0.000000,0.489925,0.331849,0.629902,0.349948,Benign,0.0,0.0,1.0
2,0.007630,0.979782,0.000034,0.000000,0.000948,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.000000,0.000000,0.489925,0.331849,0.629902,0.349948,Benign,0.0,0.0,1.0
3,0.000336,0.014545,0.000172,0.000089,0.000745,0.000013,0.098301,0.000000,0.059231,0.139603,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Benign,0.0,1.0,0.0
4,0.007630,0.745696,0.000086,0.000000,0.001896,0.000000,0.075850,0.345066,0.276311,0.000000,...,0.035348,0.035348,0.178102,0.213485,0.349938,0.060008,Benign,0.0,0.0,1.0


train data split

In [ ]:

X = df.drop(columns=['Label'])

# Train-test split on benign only
X_train, X_val = train_test_split(
    X, test_size=0.2, random_state=42
)
X_train.count(),X_val.count()

(Dst Port           838407
 Flow Duration      838407
 Tot Fwd Pkts       838407
 Tot Bwd Pkts       838407
 TotLen Fwd Pkts    838407
                     ...  
 Idle Max           838407
 Idle Min           838407
 Proto_0            838407
 Proto_6            838407
 Proto_17           838407
 Length: 80, dtype: int64,
 Dst Port           209602
 Flow Duration      209602
 Tot Fwd Pkts       209602
 Tot Bwd Pkts       209602
 TotLen Fwd Pkts    209602
                     ...  
 Idle Max           209602
 Idle Min           209602
 Proto_0            209602
 Proto_6            209602
 Proto_17           209602
 Length: 80, dtype: int64)

#Building the Model

defining the model

In [ ]:
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


Converting the data to tensors and dataloader

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Convert to torch tensors (ensure float32)
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)

# Wrap in TensorDataset and DataLoader
batch_size = 128
train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor), batch_size=batch_size, shuffle=False)


Initialization

In [ ]:
input_dim = X_train.shape[1]
model = Autoencoder(input_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


Training loop

In [ ]:
num_epochs = 10
model.train()

for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in train_loader:
        data = batch[0]
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, data)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")


Epoch [1/10], Loss: 0.0000
Epoch [2/10], Loss: 0.0000
Epoch [3/10], Loss: 0.0000
Epoch [4/10], Loss: 0.0000
Epoch [5/10], Loss: 0.0000
Epoch [6/10], Loss: 0.0000
Epoch [7/10], Loss: 0.0000
Epoch [8/10], Loss: 0.0000
Epoch [9/10], Loss: 0.0000
Epoch [10/10], Loss: 0.0000


Calculate Reconstruction Error on Validation Set

In [ ]:
model.eval()
reconstructions = []
errors = []

with torch.no_grad():
    for batch in val_loader:
        data = batch[0]
        output = model(data)
        reconstructions.append(output)
        errors.append(torch.mean((output - data) ** 2, dim=1))  # per-sample MSE

reconstructions = torch.cat(reconstructions)
errors = torch.cat(errors)


In [ ]:
threshold = torch.quantile(errors, 0.95)  # 95th percentile as threshold

Test Data

In [ ]:
X_test = df_full.drop(columns=['Label'])
y_test = df_full['Label'].values

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
test_loader = DataLoader(TensorDataset(X_test_tensor), batch_size=batch_size, shuffle=False)

model.eval()
test_errors = []

with torch.no_grad():
    for batch in test_loader:
        data = batch[0]
        output = model(data)
        error = torch.mean((output - data) ** 2, dim=1)
        test_errors.append(error)

test_errors = torch.cat(test_errors)
y_pred = (test_errors > threshold).int()


In [ ]:
print(df_full['Label'].value_counts())


Label
Benign    1048009
Name: count, dtype: int64


In [ ]:
label_map = {'Benign': 0, 'Malicious': 1}
y_test_encoded = [label_map[label] for label in y_test]


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test_encoded, y_pred, target_names=['Benign', 'Malicious']))


              precision    recall  f1-score   support

      Benign       1.00      0.95      0.97   1048009
   Malicious       0.00      0.00      0.00         0

    accuracy                           0.95   1048009
   macro avg       0.50      0.48      0.49   1048009
weighted avg       1.00      0.95      0.97   1048009



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Xval

TESTING WITH TEST DATA(only benign)

In [ ]:
X_val.value_counts()

Dst Port  Flow Duration  Tot Fwd Pkts  Tot Bwd Pkts  TotLen Fwd Pkts  TotLen Bwd Pkts  Fwd Pkt Len Max  Fwd Pkt Len Min  Fwd Pkt Len Mean  Fwd Pkt Len Std  Bwd Pkt Len Max  Bwd Pkt Len Min  Bwd Pkt Len Mean  Bwd Pkt Len Std  Flow Byts/s   Flow Pkts/s   Flow IAT Mean  Flow IAT Std  Flow IAT Max  Flow IAT Min  Fwd IAT Tot   Fwd IAT Mean  Fwd IAT Std  Fwd IAT Max   Fwd IAT Min  Bwd IAT Tot  Bwd IAT Mean  Bwd IAT Std  Bwd IAT Max  Bwd IAT Min  Fwd PSH Flags  Bwd PSH Flags  Fwd URG Flags  Bwd URG Flags  Fwd Header Len  Bwd Header Len  Fwd Pkts/s    Bwd Pkts/s    Pkt Len Min  Pkt Len Max  Pkt Len Mean  Pkt Len Std  Pkt Len Var  FIN Flag Cnt  SYN Flag Cnt  RST Flag Cnt  PSH Flag Cnt  ACK Flag Cnt  URG Flag Cnt  CWE Flag Count  ECE Flag Cnt  Down/Up Ratio  Pkt Size Avg  Fwd Seg Size Avg  Bwd Seg Size Avg  Fwd Byts/b Avg  Fwd Pkts/b Avg  Fwd Blk Rate Avg  Bwd Byts/b Avg  Bwd Pkts/b Avg  Bwd Blk Rate Avg  Subflow Fwd Pkts  Subflow Fwd Byts  Subflow Bwd Pkts  Subflow Bwd Byts  Init Fwd Win Byts  Init Bwd Win Byts  Fwd Act Data Pkts  Fwd Seg Size Min  Active Mean  Active Std  Active Max  Active Min  Idle Mean  Idle Std  Idle Max  Idle Min  Proto_0  Proto_6  Proto_17
0.001221  6.916669e-07   0.000017      0.000000      0.000000         0.000000         0.000000         0.0              0.000000          0.000000         0.000000         0.0              0.000000          0.00000          0.000000e+00  6.024092e-03  6.917150e-07   0.000000      6.917150e-07  0.000019      6.916674e-07  6.917150e-07  0.000000     6.917150e-07  0.000019     0.000000     0.000000      0.000000     0.000000     0.000000     0.0            0.0            0.0            0.0            0.000034        0.000000        6.024096e-03  0.000000e+00  0.0          0.000000     0.000000      0.000000     0.000000     0.0           0.0           0.0           0.0           1.0           0.0           0.0             0.0           0.000000       0.000000      0.000000          0.000000          0.0             0.0             0.0               0.0             0.0             0.0               0.000017          0.000000          0.000000          0.000000          0.004257           0.000000           0.000000           0.416667          0.0          0.0         0.0         0.0         0.0        0.0       0.0       0.0       0.0      1.0      0.0         392
          7.000002e-07   0.000017      0.000000      0.000000         0.000000         0.000000         0.0              0.000000          0.000000         0.000000         0.0              0.000000          0.00000          0.000000e+00  5.952377e-03  7.000489e-07   0.000000      7.000489e-07  0.000019      7.000007e-07  7.000489e-07  0.000000     7.000489e-07  0.000019     0.000000     0.000000      0.000000     0.000000     0.000000     0.0            0.0            0.0            0.0            0.000034        0.000000        5.952381e-03  0.000000e+00  0.0          0.000000     0.000000      0.000000     0.000000     0.0           0.0           0.0           0.0           1.0           0.0           0.0             0.0           0.000000       0.000000      0.000000          0.000000          0.0             0.0             0.0               0.0             0.0             0.0               0.000017          0.000000          0.000000          0.000000          0.004257           0.000000           0.000000           0.416667          0.0          0.0         0.0         0.0         0.0        0.0       0.0       0.0       0.0      1.0      0.0         373
          6.833335e-07   0.000017      0.000000      0.000000         0.000000         0.000000         0.0              0.000000          0.000000         0.000000         0.0              0.000000          0.00000          0.000000e+00  6.097557e-03  6.833811e-07   0.000000      6.833811e-07  0.000019      6.833340e-07  6.833811e-07  0.000000     6.833811e-07  0.000019     0.000000     0.000000      0.000000     0.000000     0.000000     0.0            0.0     

In [ ]:
X_test = X_val
y_test = ['Benign']*len(X_val)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
test_loader = DataLoader(TensorDataset(X_test_tensor), batch_size=batch_size, shuffle=False)

model.eval()
test_errors = []

with torch.no_grad():
    for batch in test_loader:
        data = batch[0]
        output = model(data)
        error = torch.mean((output - data) ** 2, dim=1)
        test_errors.append(error)

test_errors = torch.cat(test_errors)
y_pred = (test_errors > threshold).int()

label_map = {'Benign': 0, 'Malicious': 1}
y_test_encoded = [label_map[label] for label in y_test]

from sklearn.metrics import classification_report

print(classification_report(y_test_encoded, y_pred, target_names=['Benign', 'Malicious']))


              precision    recall  f1-score   support

      Benign       1.00      0.95      0.97    209602
   Malicious       0.00      0.00      0.00         0

    accuracy                           0.95    209602
   macro avg       0.50      0.47      0.49    209602
weighted avg       1.00      0.95      0.97    209602



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
